# 1. Imports

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pathlib import Path

# 2. Config

In [2]:
CONFIG = {
    "BASE_ROOT" : Path(r"C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_3"),
    "SAVE_PATH" : Path(r"C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_3\index.pkl"),
    
    "ROOT": Path(r"E:\WLASL\wlasl_1000_preproc"),
    "VIDEO_ROOT": Path(r"E:\WLASL\wlasl_1000_preproc\videos"),

    "TRAIN_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\train_final.json"),
    "VAL_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\val_final.json"),
    "TEST_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\test_final.json"),

    "LABEL_MAP_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\label_map_final.json"),

    "INDEX_PATH": Path(r"E:\WLASL\wlasl_1000_preproc\video_index.pkl"),

    "MAX_LEN": 48,
    "BATCH_SIZE": 16
}

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
for name, value in CONFIG.items():
    if isinstance(value, Path):
        if value.exists():
            print(f"{name} exists at {value}")
        else:
            print(f"{name} does NOT exist at {value}")
    else:
        print(f"{name} = {value}")


BASE_ROOT exists at C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_3
SAVE_PATH exists at C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_3\index.pkl
ROOT exists at E:\WLASL\wlasl_1000_preproc
VIDEO_ROOT exists at E:\WLASL\wlasl_1000_preproc\videos
TRAIN_JSON exists at E:\WLASL\wlasl_1000_preproc\train_final.json
VAL_JSON exists at E:\WLASL\wlasl_1000_preproc\val_final.json
TEST_JSON exists at E:\WLASL\wlasl_1000_preproc\test_final.json
LABEL_MAP_JSON exists at E:\WLASL\wlasl_1000_preproc\label_map_final.json
INDEX_PATH exists at E:\WLASL\wlasl_1000_preproc\video_index.pkl
MAX_LEN = 48
BATCH_SIZE = 16


# 3. Load Label Map

In [5]:
import json

with open(CONFIG["LABEL_MAP_JSON"], "r") as f:
    label_map = json.load(f)

gloss_to_idx = {v: i for i, v in enumerate(label_map.values())}
idx_to_gloss = {i: v for i, v in enumerate(label_map.values())}

print(gloss_to_idx)
print(idx_to_gloss)
NUM_CLASSES = len(gloss_to_idx)

print("Num classes:", NUM_CLASSES)

labels = list(gloss_to_idx.values())
print(min(labels), max(labels), len(set(labels)))

{'a': 0, 'a lot': 1, 'abdomen': 2, 'able': 3, 'about': 4, 'above': 5, 'accent': 6, 'accept': 7, 'accident': 8, 'accomplish': 9, 'accountant': 10, 'across': 11, 'act': 12, 'action': 13, 'active': 14, 'activity': 15, 'actor': 16, 'adapt': 17, 'add': 18, 'address': 19, 'adjective': 20, 'adjust': 21, 'admire': 22, 'admit': 23, 'adopt': 24, 'adult': 25, 'advanced': 26, 'advantage': 27, 'adverb': 28, 'affect': 29, 'afraid': 30, 'africa': 31, 'after': 32, 'afternoon': 33, 'again': 34, 'against': 35, 'age': 36, 'agenda': 37, 'ago': 38, 'agree': 39, 'agreement': 40, 'ahead': 41, 'aid': 42, 'aim': 43, 'airplane': 44, 'alarm': 45, 'alcohol': 46, 'algebra': 47, 'all': 48, 'all day': 49, 'allergy': 50, 'alligator': 51, 'allow': 52, 'almost': 53, 'alone': 54, 'alphabet': 55, 'already': 56, 'also': 57, 'always': 58, 'amazing': 59, 'america': 60, 'amputate': 61, 'analyze': 62, 'anatomy': 63, 'and': 64, 'angel': 65, 'angle': 66, 'angry': 67, 'animal': 68, 'anniversary': 69, 'announce': 70, 'annoy': 71,

# 4. JSON Parser

### Build vidId -> Label Map

In [6]:
import os
import pickle
from tqdm import tqdm

def build_index(video_root, save_path):
    if save_path.exists():
        print("Loading cached index...")
        with open(save_path, "rb") as f:
            return pickle.load(f)

    video_index = {}

    print("Building video index...")

    for root, _, files in os.walk(video_root):
        for file in files:
            if file.endswith(".npy"):
                vid_id = file.replace(".npy", "")
                full_path = os.path.join(root, file)
                video_index[vid_id] = full_path

    with open(save_path, "wb") as f:
        pickle.dump(video_index, f)

    print("Index built:", len(video_index))
    return video_index

video_index = build_index(CONFIG["VIDEO_ROOT"], CONFIG["INDEX_PATH"])
print("Indexed videos:", len(video_index))

Loading cached index...
Indexed videos: 6073


In [7]:
print(list(video_index.keys())[:5])

['01610', '01612', '01615', '66039', '00663']


vid_id --> Label

In [8]:
def parse_split(json_path, gloss_to_idx, video_index):
    import json

    with open(json_path, "r") as f:
        data = json.load(f)

    samples = []
    missing_video = 0

    for entry in data:
        gloss = entry["gloss"]

        if gloss not in gloss_to_idx:
            continue

        label = gloss_to_idx[gloss]

        for inst in entry["instances"]:
            vid = inst["video_id"]

            if vid not in video_index:
                missing_video += 1
                continue

            samples.append((vid, label))

    print(f"{json_path.name} → kept {len(samples)}, removed {missing_video}")
    return samples

# 5. Dataset

In [9]:
import numpy as np
import torch
from torch.utils.data import Dataset

class SignDataset(Dataset):
    def __init__(self, samples, video_index, max_len=48):
        self.samples = samples
        self.video_index = video_index
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # ---- Safe loading ----
        for _ in range(5):
            vid, label = self.samples[idx]

            if vid in self.video_index:
                path = self.video_index[vid]
                try:
                    data = np.load(path)  # (T,75,3) or (T,75,2)
                    break
                except:
                    pass

            idx = (idx + 1) % len(self.samples)
        else:
            raise RuntimeError("Too many failed data loads")

        # ---- Drop Z ----
        if data.ndim == 3:
            data = data[..., :2]  # (T,75,2)

        # ---- Normalize (center + scale) ----
        left_sh = data[:, 11]   # (T,2)
        right_sh = data[:, 12]  # (T,2)

        center = (left_sh + right_sh) / 2.0
        data = data - center[:, None, :]

        scale = np.linalg.norm(left_sh - right_sh, axis=1, keepdims=True) + 1e-6
        data = data / scale[:, None, :]

        # ---- Velocity ----
        velocity = data[1:] - data[:-1]  # (T-1,75,2)

        # pad last frame to keep same length
        velocity = np.concatenate(
            [velocity, np.zeros_like(velocity[:1])],
            axis=0
        )  # (T,75,2)

        # ---- Concatenate position + velocity ----
        data = np.concatenate([data, velocity], axis=-1)  # (T,75,4)

        # ---- Flatten ----
        T = data.shape[0]
        data = data.reshape(T, -1)  # (T, 75*4 = 300)

        length = min(T, self.max_len)
        
        # --- add noise ---
        noise = np.random.normal(0, 0.01, data.shape)
        data += noise

        # ---- Pad / truncate ----
        if T > self.max_len:
            data = data[:self.max_len]
        else:
            pad = np.zeros((self.max_len - T, data.shape[1]), dtype=data.dtype)
            data = np.concatenate([data, pad], axis=0)

        return (
            torch.tensor(data, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(length, dtype=torch.long)
        )

# 6. Dataloaders

In [10]:
train_samples = parse_split(CONFIG["TRAIN_JSON"], gloss_to_idx, video_index)
val_samples = parse_split(CONFIG["VAL_JSON"], gloss_to_idx, video_index)
test_samples = parse_split(CONFIG["TEST_JSON"], gloss_to_idx, video_index)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))

train_final.json → kept 4204, removed 2987
val_final.json → kept 1135, removed 835
test_final.json → kept 734, removed 723
Train: 4204
Val: 1135
Test: 734


In [11]:
print("Train coverage:", len(train_samples) / (len(train_samples) + 2987))

missing = []
for entry in json.load(open(CONFIG["TRAIN_JSON"])):
    for inst in entry["instances"]:
        vid = inst["video_id"]
        if vid not in video_index:
            missing.append(vid)

print(missing[:20])
print(list(video_index.keys())[:20])

Train coverage: 0.5846196634682241
['65225', '68011', '68208', '68012', '70266', '07085', '07086', '07087', '07088', '07089', '07090', '07091', '07097', '07071', '07073', '67424', '07075', '07076', '07077', '07078']
['01610', '01612', '01615', '66039', '00663', '00664', '00666', '00668', '65010', '03490', '03491', '03493', '65098', '32510', '32511', '32512', '32514', '32518', '66019', '03515']


In [12]:
def collate_fn(batch):
    data, labels, lengths = zip(*batch)

    data = torch.stack(data)
    labels = torch.stack(labels)
    lengths = torch.stack(lengths)

    return data, labels, lengths

In [13]:
from torch.utils.data import DataLoader

train_dataset = SignDataset(train_samples, video_index, CONFIG["MAX_LEN"])
val_dataset = SignDataset(val_samples, video_index, CONFIG["MAX_LEN"])
test_dataset = SignDataset(test_samples, video_index, CONFIG["MAX_LEN"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    collate_fn=collate_fn
)

In [14]:
sample = next(iter(train_loader))[0]   # (B, T, D)
INPUT_DIM = sample.shape[-1]
print(INPUT_DIM)

300


# 7. Utils

In [15]:
import numpy as np

def lm_to_np(lms, n):
    if lms is None:
        return np.zeros((n, 2), dtype=np.float32)

    coords = np.array(
        [[lm.x, lm.y] for lm in lms.landmark],
        dtype=np.float32
    )

    if coords.shape[0] != n:
        out = np.zeros((n, 2), dtype=np.float32)
        out[:coords.shape[0]] = coords
        return out

    return coords

def extract_75_xy(results):
    pose = lm_to_np(results.pose_landmarks, 33)[:, :2]
    left = lm_to_np(results.left_hand_landmarks, 21)[:, :2]
    right = lm_to_np(results.right_hand_landmarks, 21)[:, :2]
    return np.concatenate([pose, left, right], axis=0)  # (75,2)

def normalize_landmarks(frame_lm):
    # center
    left_sh = frame_lm[11]
    right_sh = frame_lm[12]
    center = (left_sh + right_sh) / 2.0
    frame_lm = frame_lm - center

    # scale 
    scale = np.linalg.norm(left_sh - right_sh) + 1e-6
    frame_lm = frame_lm / scale

    return frame_lm

In [16]:
from collections import Counter
import torch

def compute_class_weights(samples, num_classes):
    labels = [label for _, label in samples]
    counts = Counter(labels)

    total = sum(counts.values())

    weights = []
    for i in range(num_classes):
        if i in counts:
            weights.append(total / counts[i])
        else:
            weights.append(0.0)  # class not present

    weights = torch.tensor(weights, dtype=torch.float32)
    return weights, counts

In [17]:
import pickle

def save_class_weights(weights, counts, path):
    with open(path, "wb") as f:
        pickle.dump({
            "weights": weights,
            "counts": counts
        }, f)

In [18]:
def load_class_weights(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["weights"], data["counts"]

## Build Class Weights

In [19]:
weights, counts = compute_class_weights(train_samples, NUM_CLASSES)
weights = weights.to(device)
save_path = CONFIG["BASE_ROOT"] / "class_weights.pkl"
save_class_weights(weights, counts, save_path)

print("Saved class weights to:", save_path)

Saved class weights to: C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_3\class_weights.pkl


## Other Utils

In [20]:
SAVE_PATH = CONFIG["BASE_ROOT"]

PLOTS_PATH = SAVE_PATH / "plots"
METRICS_PATH = SAVE_PATH / "metrics"
CONF_PATH = SAVE_PATH / "confusion"

for p in [SAVE_PATH, PLOTS_PATH, METRICS_PATH, CONF_PATH]:
    p.mkdir(parents=True, exist_ok=True)

### metrics functions

In [21]:
import csv

METRICS_FILE = METRICS_PATH / "metrics.csv"

def init_metrics_file():
    with open(METRICS_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss", "val_loss",
            "train_acc", "val_acc"
        ])

def log_metrics(epoch, train_loss, val_loss, train_acc, val_acc):
    with open(METRICS_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            train_loss, val_loss,
            train_acc, val_acc
        ])

### lost + acc plots

In [22]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_metrics():
    df = pd.read_csv(METRICS_FILE)

    # Loss Plot
    plt.figure()
    plt.plot(df["epoch"], df["train_loss"], label="Train Loss")
    plt.plot(df["epoch"], df["val_loss"], label="Val Loss")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.savefig(PLOTS_PATH / "loss.png")
    plt.close()

    # Accuracy Plot
    plt.figure()
    plt.plot(df["epoch"], df["train_acc"], label="Train Acc")
    plt.plot(df["epoch"], df["val_acc"], label="Val Acc")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.savefig(PLOTS_PATH / "accuracy.png")
    plt.close()

In [23]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def compute_comprehensive_metrics(
    all_preds_top1,
    all_labels,
    num_classes,
    epoch,
    phase='Val',
    all_preds_top5=None
):
    """
    Compute classification metrics for multi-class problem (WLASL-style)
    """

    print(f"\n{'='*60}")
    print(f"📊 {phase} METRICS - Epoch {epoch}")
    print(f"{'='*60}")

    # ---------------- TOP-1 ----------------
    accuracy = accuracy_score(all_labels, all_preds_top1)

    # ---------------- TOP-5 ----------------
    if all_preds_top5 is not None:
        top5_accuracy = float(np.mean(all_preds_top5))
    else:
        top5_accuracy = None
        

    # ---------------- PRECISION / RECALL / F1 ----------------
    precision_macro = precision_score(
        all_labels, all_preds_top1, average='macro', zero_division=0
    )
    recall_macro = recall_score(
        all_labels, all_preds_top1, average='macro', zero_division=0
    )
    f1_macro = f1_score(
        all_labels, all_preds_top1, average='macro', zero_division=0
    )

    precision_weighted = precision_score(
        all_labels, all_preds_top1, average='weighted', zero_division=0
    )
    recall_weighted = recall_score(
        all_labels, all_preds_top1, average='weighted', zero_division=0
    )
    f1_weighted = f1_score(
        all_labels, all_preds_top1, average='weighted', zero_division=0
    )

    # ---------------- CONFUSION MATRIX ----------------
    cm = confusion_matrix(
        all_labels,
        all_preds_top1,
        labels=np.arange(num_classes)  # ensures fixed size matrix
    )

    # ---------------- PRINT ----------------
    print(f"\n📈 Overall:")
    print(f"   Top-1 Accuracy: {accuracy*100:.2f}%")

    if top5_accuracy is not None:
        print(f"   Top-5 Accuracy: {top5_accuracy*100:.2f}%")

    print(f"\n📊 Macro:")
    print(f"   Precision: {precision_macro*100:.2f}%")
    print(f"   Recall:    {recall_macro*100:.2f}%")
    print(f"   F1 Score:  {f1_macro*100:.2f}%")

    print(f"\n📊 Weighted:")
    print(f"   Precision: {precision_weighted*100:.2f}%")
    print(f"   Recall:    {recall_weighted*100:.2f}%")
    print(f"   F1 Score:  {f1_weighted*100:.2f}%")

    print(f"\n📊 Confusion Matrix Stats:")
    print(f"   Correct Predictions: {np.trace(cm)}")
    print(f"   Total Predictions:  {cm.sum()}")

    # ---------------- RETURN ----------------
    return {
        "accuracy": accuracy,
        "top5_accuracy": top5_accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "confusion_matrix": cm
    }

### confusion metrics

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(
    cm,
    epoch,
    phase,
    save_path,
    num_classes=None,
    top_k=50,
    normalize=False,
    class_names=None,
    save_raw=True
):
    """
    Plot and save confusion matrix.

    Args:
        cm: numpy array (num_classes x num_classes)
        epoch: current epoch
        phase: 'Train' / 'Val' / 'Test'
        save_path: file path (image)
        num_classes: total number of classes (ensures fixed size)
        top_k: show only top-k most frequent classes
        normalize: normalize rows
        class_names: optional list of class names
        save_raw: save raw cm as .npy
    """

    cm = np.array(cm)

    # ---------------- FIX SIZE ----------------
    if num_classes is not None and cm.shape[0] != num_classes:
        fixed_cm = np.zeros((num_classes, num_classes), dtype=cm.dtype)
        fixed_cm[:cm.shape[0], :cm.shape[1]] = cm
        cm = fixed_cm

    # ---------------- NORMALIZE ----------------
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = cm.astype(float) / (row_sums + 1e-8)

    # ---------------- TOP-K REDUCTION ----------------
    num_classes = cm.shape[0]

    if num_classes > top_k:
        row_sums = cm.sum(axis=1)

        # take most frequent true classes
        top_indices = np.argsort(row_sums)[-top_k:]
        cm = cm[np.ix_(top_indices, top_indices)]

        if class_names is not None:
            class_names = [class_names[i] for i in top_indices]

        title_suffix = f"(Top {top_k} classes)"
    else:
        title_suffix = "(All classes)"

    # ---------------- PLOT ----------------
    plt.figure(figsize=(12, 10))

    sns.heatmap(
        cm,
        cmap="Blues",
        cbar=True,
        xticklabels=class_names if class_names is not None else False,
        yticklabels=class_names if class_names is not None else False
    )

    plt.title(f"{phase} Confusion Matrix - Epoch {epoch} {title_suffix}", fontsize=14)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    # ---------------- SAVE RAW ----------------
    if save_raw:
        raw_path = str(save_path).replace(".png", ".npy")
        np.save(raw_path, cm)

    print(f"💾 Confusion matrix saved → {save_path}")

In [25]:
import matplotlib.pyplot as plt

def plot_metrics_history(history, save_path):
    """
    Plot training history curves.

    history dict keys expected:
    - train_loss, val_loss
    - train_acc, val_acc
    - val_top5 (optional)
    """

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(15, 10))

    # ---------------- LOSS ----------------
    plt.subplot(2, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train")
    plt.plot(epochs, history["val_loss"], label="Val")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    # ---------------- ACCURACY ----------------
    plt.subplot(2, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train")
    plt.plot(epochs, history["val_acc"], label="Val")
    plt.title("Top-1 Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    # ---------------- TOP-5 ----------------
    if "val_top5" in history:
        plt.subplot(2, 2, 3)
        plt.plot(epochs, history["val_top5"], label="Val Top-5")
        plt.title("Top-5 Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(alpha=0.3)

    # ---------------- GAP (overfitting detector) ----------------
    plt.subplot(2, 2, 4)
    gap = [v - t for v, t in zip(history["val_acc"], history["train_acc"])]
    plt.plot(epochs, gap)
    plt.title("Generalization Gap (Val - Train)")
    plt.xlabel("Epoch")
    plt.ylabel("Gap")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"💾 Training curves saved → {save_path}")

### normalized heatmap

In [26]:
# def save_confusion_heatmap(y_true, y_pred, epoch, top_k=50):
#     cm = confusion_matrix(
#         y_true,
#         y_pred,
#         labels=np.arange(NUM_CLASSES)
#     )

#     # Normalize
#     cm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-6)

#     # Reduce to top-K frequent classes
#     row_sums = cm.sum(axis=1)
#     top_indices = np.argsort(row_sums)[-top_k:]
#     cm = cm[np.ix_(top_indices, top_indices)]

#     plt.figure(figsize=(10, 8))
#     plt.imshow(cm, interpolation='nearest')
#     plt.title(f"Normalized CM (Top {top_k}) Epoch {epoch}")
#     plt.colorbar()

#     plt.xlabel("Predicted")
#     plt.ylabel("True")

#     plt.savefig(CONF_PATH / f"cm_norm_epoch_{epoch}.png")
#     plt.close()

### precision recall f1

In [27]:
from sklearn.metrics import classification_report

def save_classification_report(y_true, y_pred, epoch):
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        output_dict=True,
        zero_division=0
    )

    import json
    with open(CONF_PATH / f"report_epoch_{epoch}.json", "w") as f:
        json.dump(report, f, indent=4, default=float)
    

### evaluation

In [28]:
def evaluate_model(model, loader, device, criterion, num_classes, epoch, phase='Val'):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0
    batches = 0
    all_top5 = []

    with torch.no_grad():
        for seqs, lbls, lengths in tqdm(loader, desc=f"{phase} Evaluation", ncols=100):

            seqs = seqs.to(device)
            lbls = lbls.to(device)
            lengths = lengths.to(device)

            outputs = model(seqs, lengths)
            loss = criterion(outputs, lbls)

            total_loss += loss.item()

            # Top-1
            top1_preds = outputs.argmax(1).cpu().numpy()

            # Top-5 (correct format already)
            top5 = torch.topk(outputs, k=5, dim=1).indices
            correct_top5 = top5.eq(lbls.unsqueeze(1)).any(dim=1)
            top5_preds = correct_top5.cpu().numpy().astype(int)

            all_preds.extend(top1_preds)
            all_top5.extend(top5_preds)
            all_labels.extend(lbls.cpu().numpy())

            batches += 1

    avg_loss = total_loss / max(1, batches)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_top5 = np.array(all_top5)

    metrics = compute_comprehensive_metrics(
        all_preds,
        all_labels,
        num_classes,
        epoch,
        phase,
        all_preds_top5=all_top5
    )

    metrics["loss"] = avg_loss
    metrics["all_preds"] = all_preds
    metrics["all_labels"] = all_labels

    return metrics

# 8. Model

In [29]:
import torch
import torch.nn as nn
import math
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTMTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=384,
        nhead=8,
        num_lstm_layers=2,
        num_transformer_layers=2,
        max_len=512,
        dropout=0.3
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        # 1 - Frame Encoder
        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # 2 - Positional Encoding 
        self.register_buffer("pos_encoding", self._build_pos_encoding(max_len, hidden_dim))

        # 3 - BiLSTM
        self.lstm = nn.LSTM(
            hidden_dim,
            hidden_dim // 2,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )

        # 4 - Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True  
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers
        )

        # 5 - Attention Pooling 
        self.attn_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # 6 - Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

        self._init_weights()

    # Positional Encoding
    # ---------------------------
    def _build_pos_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        return pe.unsqueeze(0)  # (1, T, D)

    # Masked Attention Pooling
    # -------------------------
    def attention_pool(self, x, mask=None):
        # x: (B, T, H)

        scores = self.attn_pool(x)  # (B, T, 1)

        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(-1), -1e9)

        weights = torch.softmax(scores, dim=1)
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)
        pooled = torch.sum(weights * x, dim=1)

        return pooled

    # Forward
    # -----------------
    def forward(self, x, lengths=None):
        B, T, _ = x.shape
        device = x.device

        # Build padding mask
        if lengths is not None:
            mask = torch.arange(T, device=device).expand(B, T) >= lengths.unsqueeze(1)
        else:
            mask = None

        x = self.frame_encoder(x)

        x = x + self.pos_encoding[:, :T, :].to(device)

        # LSTM with packing
        if lengths is not None:
            from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
            x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            x, _ = self.lstm(x)
            x, _ = pad_packed_sequence(x, batch_first=True, total_length=T)
        else:
            x, _ = self.lstm(x)

        x = self.transformer(x, src_key_padding_mask=mask)

        x = self.attention_pool(x, mask)

        return self.classifier(x)

    # Weight Init
    # ----------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

In [30]:
def sanity_check_overfit(model_class, model_args, train_loader, device, max_epochs=200):
    print(f"\n{'='*60}\n == SANITY CHECK: Attempting to overfit a single batch\n{'='*60}")
    model = model_class(**model_args).to(device)

    single_batch = None
    for seqs, lbls, lengths in train_loader:
        single_batch = (seqs, lbls, lengths)
        break

    if single_batch is None:
        print("Could not load a valid batch!")
        return False
    
    seqs, lbls, lengths = single_batch
    seqs = seqs.to(device)
    lbls = lbls.to(device)
    lengths = lengths.to(device)

    print(f"   Batch size: {seqs.shape[0]}, Unique labels: {len(torch.unique(lbls))}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        outputs = model(seqs, lengths)
        loss = criterion(outputs, lbls)

        if torch.isnan(loss).any().item():
            print("NaN loss detected; skipping step.")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            acc = (outputs.argmax(1) == lbls).float().mean().item() * 100
            print(f"   Epoch {epoch+1:3d}: Loss={loss.item():.4f}, Acc={acc:.2f}%")

            if acc > 95:
                print(f"\n == > Overfitted in {epoch+1} epochs. Model can learn.")
                return True
    
    print(f"\n  ! == > Could not overfit. There is a fundamental issue.")
    return False

# 9. Train

In [31]:
def run_training_pipeline(
    model_class,
    model_args,
    train_loader,
    val_loader,
    test_loader,
    num_classes,
    save_dir,
    device="cuda",
    epochs=100
):
    import os
    import torch
    import json

    os.makedirs(save_dir, exist_ok=True)

    BEST_MODEL_PATH = os.path.join(save_dir, "best_model.pth")
    FINAL_RESULTS_PATH = os.path.join(save_dir, "final_test_results.json")

    # 1- Sanity Check
    print("\n == Running sanity check...")
    if not sanity_check_overfit(model_class, model_args, train_loader, device):
        print(" == Sanity check failed. Fix model/data first.")
        return

    print(" == > Sanity check passed.\n")

    # 2 - Init Model
    model = model_class(**model_args).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=0.1
    )

    best_val_acc = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "val_top5": []
    }

    patience = 10
    best_epoch = 0

    # 3 - TRAIN LOOP
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f" -- Epoch {epoch+1}/{epochs}")
        print(f"{'='*60}")

        # ---- TRAIN ----
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for x, y, lengths in train_loader:

            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            optimizer.zero_grad()

            out = model(x, lengths)

            loss = criterion(out, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()

            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total

        # ---- VALIDATION ----
        val_metrics = evaluate_model(
            model, val_loader, device,
            criterion, num_classes, epoch+1, "Val"
        )

        val_acc = val_metrics["accuracy"]
        val_loss = val_metrics["loss"]

        # ---- SAVE HISTORY ----
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_top5"].append(val_metrics["top5_accuracy"])

        print(f"\n📊 Epoch Summary:")
        print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}%")
        print(f"Val   Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%")

        scheduler.step()

        # ---- SAVE BEST ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_acc": val_acc
            }, BEST_MODEL_PATH)

            print("✅ Saved new best model!")

            path_raw = os.path.join(save_dir, f"cm_epoch_raw_{epoch+1}.png")
            path_norm = os.path.join(save_dir, f"cm_epoch_norm_{epoch+1}.png")
            
            plot_confusion_matrix(val_metrics["confusion_matrix"], epoch+1, "Val", path_raw, normalize=False)
            plot_confusion_matrix(val_metrics["confusion_matrix"], epoch+1, "Val", path_norm, normalize=True)
        else:
            if epoch - best_epoch > patience:
                print("⛔ Early stopping triggered")
                break

    # -------------------------
    # 4.TRAINING DONE
    # -------------------------
    print("\n🏁 Training complete.")
    plot_metrics_history(history, os.path.join(save_dir, "training_curves.png"))

    # -------------------------
    # 5. LOAD BEST MODEL
    # -------------------------
    checkpoint = torch.load(BEST_MODEL_PATH)
    model.load_state_dict(checkpoint["model_state_dict"])

    # -------------------------
    # 6. TEST EVALUATION
    # -------------------------
    print("\n🧪 Running FINAL TEST evaluation...")

    test_metrics = evaluate_model(
        model, test_loader, device,
        criterion, num_classes, epoch="FINAL", phase="Test"
    )

    print("\n🎯 FINAL TEST RESULTS:")
    print(f"Top-1 Accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Top-5 Accuracy: {test_metrics['top5_accuracy']*100:.2f}%")
    print(f"F1 Macro: {test_metrics['f1_macro']*100:.2f}%")

    # Save results
    with open(FINAL_RESULTS_PATH, "w") as f:
        json.dump({
            "top1_accuracy": test_metrics["accuracy"],
            "top5_accuracy": test_metrics["top5_accuracy"],
            "f1_macro": test_metrics["f1_macro"]
        }, f, indent=4)

    print(f"\n💾 Results saved at: {FINAL_RESULTS_PATH}")

    return model, history, test_metrics

In [32]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_args = {
    "input_dim": INPUT_DIM,
    "num_classes": NUM_CLASSES,
    "hidden_dim": 256,
    "nhead": 8,
    "num_lstm_layers": 2,
    "num_transformer_layers": 1
}

model, history, test_metrics = run_training_pipeline(
    model_class=BiLSTMTransformerModel,
    model_args=model_args,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_classes=NUM_CLASSES,
    save_dir="./outputs/bilstm_transformer_run5",
    device=device,
    epochs=100
)


 == Running sanity check...

 == SANITY CHECK: Attempting to overfit a single batch


c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\torch\nn\modules\transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


   Batch size: 16, Unique labels: 16


c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\torch\nn\functional.py:5504: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


   Epoch  20: Loss=3.6614, Acc=43.75%
   Epoch  40: Loss=1.8771, Acc=93.75%
   Epoch  60: Loss=0.5502, Acc=100.00%

 == > Overfitted in 60 epochs. Model can learn.
 == > Sanity check passed.


 -- Epoch 1/100


...

Val Evaluation: 100%|███████████████████████████████████████████████| 71/71 [00:04<00:00, 15.51it/s]
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarni


📊 Val METRICS - Epoch 92

📈 Overall:
   Top-1 Accuracy: 10.13%
   Top-5 Accuracy: 26.43%

📊 Macro:
   Precision: 6.96%
   Recall:    8.22%
   F1 Score:  6.93%

📊 Weighted:
   Precision: 9.57%
   Recall:    10.13%
   F1 Score:  9.03%

📊 Confusion Matrix Stats:
   Correct Predictions: 115
   Total Predictions:  1135

📊 Epoch Summary:
Train Loss: 1.7978, Acc: 91.75%
Val   Loss: 6.1202, Acc: 10.13%
⛔ Early stopping triggered

🏁 Training complete.
💾 Training curves saved → ./outputs/bilstm_transformer_run5\training_curves.png

🧪 Running FINAL TEST evaluation...


Test Evaluation: 100%|██████████████████████████████████████████████| 46/46 [00:19<00:00,  2.32it/s]


📊 Test METRICS - Epoch FINAL

📈 Overall:
   Top-1 Accuracy: 8.45%
   Top-5 Accuracy: 26.29%

📊 Macro:
   Precision: 5.64%
   Recall:    6.48%
   F1 Score:  5.61%

📊 Weighted:
   Precision: 8.17%
   Recall:    8.45%
   F1 Score:  7.66%

📊 Confusion Matrix Stats:
   Correct Predictions: 62
   Total Predictions:  734

🎯 FINAL TEST RESULTS:
Top-1 Accuracy: 8.45%
Top-5 Accuracy: 26.29%
F1 Macro: 5.61%

💾 Results saved at: ./outputs/bilstm_transformer_run5\final_test_results.json



c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of

In [ ]:
import hashlib

target_hash = "9edf6713ae21b1a80951c3e9aa069a0213a7a4eb"

wordlist = ["password", "admin", "hacker", "hello", "india"]

for word in wordlist:
    hashed = hashlib.sha1(word.encode()).hexdigest()
    
    if hashed == target_hash:
        print("Found:", word)
        break

In [ ]:
import torch
import json
import os

def run_test(
    model_class,
    model_args,
    checkpoint_path,
    test_loader,
    device,
    num_classes,
    save_dir
):
    os.makedirs(save_dir, exist_ok=True)

    print("\n🧪 Running FINAL TEST...")

    # -----------------------
    # 🔹 Load Model
    # -----------------------
    model = model_class(**model_args).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    print(f"✅ Loaded model from: {checkpoint_path}")

    # -----------------------
    # 🔹 Loss Function
    # -----------------------
    criterion = torch.nn.CrossEntropyLoss()

    # -----------------------
    # 🔹 Evaluate
    # -----------------------
    test_metrics = evaluate_model(
        model,
        test_loader,
        device,
        criterion,
        num_classes,
        epoch="FINAL",
        phase="Test"
    )

    # -----------------------
    # 🔹 Print Results
    # -----------------------
    print("\n🎯 FINAL TEST RESULTS")
    print(f"Top-1 Accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Top-5 Accuracy: {test_metrics['top5_accuracy']*100:.2f}%")
    print(f"F1 Macro:       {test_metrics['f1_macro']*100:.2f}%")

    # -----------------------
    # 🔹 Save Confusion Matrix
    # -----------------------
    cm = confusion_matrix(
        test_metrics["all_labels"],
        test_metrics["all_preds"],
        labels=np.arange(NUM_CLASSES)
    )

    plot_confusion_matrix(
        cm,
        epoch="test",
        phase="Test",
        save_path=save_dir / "cm_test.png",
        num_classes=NUM_CLASSES
    )

    plot_confusion_matrix(
        cm,
        epoch="test",
        phase="Test",
        save_path=save_dir / "cm_test_norm.png",
        num_classes=NUM_CLASSES,
        normalize=True
    )

    # -----------------------
    # 🔹 Save Results JSON
    # -----------------------
    results = {
        "top1_accuracy": float(test_metrics["accuracy"]),
        "top5_accuracy": float(test_metrics["top5_accuracy"]),
        "f1_macro": float(test_metrics["f1_macro"]),
        "precision_macro": float(test_metrics["precision_macro"]),
        "recall_macro": float(test_metrics["recall_macro"])
    }

    with open(os.path.join(save_dir, "test_results.json"), "w") as f:
        json.dump(results, f, indent=4)

    print(f"\n💾 Test results saved → {save_dir}/test_results.json")

    return test_metrics

In [ ]:
TEST_SAVE_DIR = "./outputs/test_results"

test_metrics = run_test(
    model_class=BiLSTMTransformerModel,
    model_args=model_args,
    checkpoint_path="./outputs/best_model.pth",
    test_loader=test_loader,
    device="cuda",
    num_classes=NUM_CLASSES,
    save_dir=TEST_SAVE_DIR
)